In [ ]:
# In colab run this cell first to setup the file structure!
%cd /content
!rm -rf MOL518-Intro-to-Data-Analysis

!git clone https://github.com/shaevitz/MOL518-Intro-to-Data-Analysis.git
%cd MOL518-Intro-to-Data-Analysis/Lecture_35

# BPY518 Lecture 35: Particle Tracking

## Lecture Outline
- Spot detection and the particle-tracking pipeline
- LoG filtering and local peak finding
- Subpixel localization by centroid and Gaussian fitting
- Linking particles across frames
- Nearest-neighbor and Hungarian assignment
- When to use richer temporal models or optical flow

Particle tracking is a standard workflow in microscopy when objects appear as isolated bright spots: single molecules, small fluorescent complexes, FISH puncta, or diffraction-limited particles.

Here is a typical sparse fluorescence field:

![Typical fluorescence field](media/particle_field.png)

A high-level pipeline for tracking is:

1. Preprocess the image.
2. Find candidate peaks at pixel resolution.
3. Refine positions to subpixel accuracy.
4. Link detections across time.

We will go through these steps one by one.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, spatial
from scipy.optimize import curve_fit, linear_sum_assignment

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'

media_dir = Path('media')
rng = np.random.default_rng(7)

## A Single Particle Looks Like a Local Intensity Peak

A diffraction-limited emitter does not look like a single bright pixel. Its light is spread out by the microscope point-spread function (PSF), so each particle appears as a small blurry mound of intensity.

![3D surface of a single particle](media/single_molecule_surface.png)

For this reason, we talk about particle localization as an *estimation problem*: given a noisy mound of intensity, what is our best guess as to the location of the center?

In the code cell below, we define several functions that will be useful as we go through the lecture. These are:

- `make_spot_image(...)`: generate a synthetic image with Gaussian-like spots, background, and optional noise.
- `find_local_maxima(...)`: detect candidate particle locations from local intensity peaks.
- `centroid_localization(...)`: estimate subpixel position using an intensity-weighted centroid in a local window.
- `gaussian_2d(...)`: model function for a 2D Gaussian plus constant background.
- `fit_gaussian_2d(...)`: fit the local image patch to the Gaussian model and return best-fit parameters.
- `compute_cost_matrix(...)`: compute pairwise distances between detections in two frames for particle linking/tracking.

In [ ]:
def make_spot_image(shape, positions, amplitudes=None, sigma=1.8, background=20.0, noise=3.0, rng=None):
    yy, xx = np.mgrid[: shape[0], : shape[1]]
    image = np.full(shape, background, dtype=float)
    if amplitudes is None:
        amplitudes = np.full(len(positions), 180.0)
    for (y0, x0), amp in zip(positions, amplitudes):
        image += amp * np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * sigma**2))
    if rng is not None:
        image += noise * rng.normal(size=shape)
    return np.clip(image, 0, None)


def find_local_maxima(image, min_distance=4, threshold=0.0):
    size = 2 * min_distance + 1
    local_max = ndimage.maximum_filter(image, size=size, mode='reflect')
    peaks = np.argwhere((image == local_max) & (image > threshold))
    return peaks


def centroid_localization(image, y, x, radius=4):
    y_min = max(0, y - radius)
    y_max = min(image.shape[0], y + radius + 1)
    x_min = max(0, x - radius)
    x_max = min(image.shape[1], x + radius + 1)
    window = image[y_min:y_max, x_min:x_max]
    weights = window - np.min(window)
    total_intensity = np.sum(weights)
    if total_intensity <= 0:
        return np.array([float(y), float(x)])
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    y_centroid = np.sum(yy * weights) / total_intensity
    x_centroid = np.sum(xx * weights) / total_intensity
    return np.array([y_centroid, x_centroid])


def gaussian_2d(coords, A, x0, y0, sigma, B):
    x, y = coords
    return A * np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2 * sigma**2)) + B


def fit_gaussian_2d(image, y, x, radius=5):
    y_min = max(0, y - radius)
    y_max = min(image.shape[0], y + radius + 1)
    x_min = max(0, x - radius)
    x_max = min(image.shape[1], x + radius + 1)
    patch = image[y_min:y_max, x_min:x_max]
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    baseline = float(np.median(patch))
    amplitude = float(np.max(patch) - baseline)
    p0 = (amplitude, float(x), float(y), 2.0, baseline)
    bounds = (
        [0.0, x_min, y_min, 0.5, 0.0],
        [np.inf, x_max, y_max, 6.0, np.inf],
    )
    popt, _ = curve_fit(
        gaussian_2d,
        (xx.ravel(), yy.ravel()),
        patch.ravel(),
        p0=p0,
        bounds=bounds,
        maxfev=2000,
    )
    return {
        'amplitude': popt[0],
        'x': popt[1],
        'y': popt[2],
        'sigma': popt[3],
        'background': popt[4],
        'patch': patch,
        'xx': xx,
        'yy': yy,
    }


def compute_cost_matrix(points_t, points_t1):
    # Compute the cost matrix based on Euclidean distance between points in frame t and t+1.
    cost_matrix = np.zeros((len(points_t), len(points_t1)))
    for i in range(len(points_t)):
        for j in range(len(points_t1)):
            dy = points_t[i, 0] - points_t1[j, 0]
            dx = points_t[i, 1] - points_t1[j, 1]
            cost_matrix[i, j] = np.sqrt(dx**2 + dy**2)
    return cost_matrix

## Preprocessing and LoG Filtering

Several preprocessing options are reasonable, including Gaussian smoothing, bandpass filtering, median filtering, and the **Laplacian of Gaussian (LoG)**. For spot detection, LoG filtering is a common choice because it emphasizes small blob-like features while suppressing smooth background structure.

We will demonstrate it on a synthetic image of fluorescent spots.

In [ ]:
# Create a synthetic field of fluorescent spots and filter it with a LoG kernel.
demo_positions = np.array([
    [24.5, 26.0],
    [32.8, 72.4],
    [58.2, 42.7],
    [72.4, 86.5],
    [83.0, 20.5],
])
demo_amplitudes = np.array([210, 180, 240, 195, 225], dtype=float)
demo_image = make_spot_image((100, 100), demo_positions, amplitudes=demo_amplitudes, sigma=1.7, background=18, noise=3.5, rng=rng)
log_filtered = -ndimage.gaussian_laplace(demo_image, sigma=1.6)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(demo_image)
axes[0].set_title('Synthetic raw image')
axes[0].axis('off')

im = axes[1].imshow(log_filtered, cmap='magma')
axes[1].set_title('LoG-filtered image')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()

## Local Peak Finding with a Maximum Filter

After filtering, we still need to decide which pixels are candidate particles. A standard rule is:

- a peak must be brighter than its neighbors in a local window,
- and it must exceed a threshold.

A maximum filter makes that easy: every pixel is replaced by the maximum value in its neighborhood. Pixels that are unchanged by this operation and also exceed the threshold are local maxima.

The output of the next cell should help you see how this works.

In [ ]:
toy_image = np.array(
    [
        [0, 0, 1, 0, 0, 0, 0],
        [0, 2, 3, 2, 1, 0, 0],
        [1, 3, 5, 4, 2, 1, 0],
        [0, 2, 4, 9, 3, 1, 0],
        [0, 1, 2, 3, 2, 1, 0],
        [0, 0, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=float,
)

toy_max = ndimage.maximum_filter(toy_image, size=3, mode='nearest')
center_y, center_x = 3, 3
window = toy_image[center_y - 1 : center_y + 2, center_x - 1 : center_x + 2]
window_max = np.max(window)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

for ax, image, title in zip(
    axes,
    [toy_image, window, toy_max],
    ['Input image', '3x3 neighborhood', 'Maximum-filter output'],
):
    ax.imshow(image, cmap='magma', vmin=0, vmax=window_max)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)
        spine.set_edgecolor('0.2')

for y in range(toy_image.shape[0]):
    for x in range(toy_image.shape[1]):
        color = 'black' if toy_image[y, x] > 0.7 * window_max else 'white'
        axes[0].text(x, y, f'{toy_image[y, x]:.0f}', ha='center', va='center', color=color, fontsize=9)
        color = 'black' if toy_max[y, x] > 0.7 * window_max else 'white'
        axes[2].text(x, y, f'{toy_max[y, x]:.0f}', ha='center', va='center', color=color, fontsize=9)

for y in range(window.shape[0]):
    for x in range(window.shape[1]):
        color = 'black' if window[y, x] > 0.7 * window_max else 'white'
        axes[1].text(x, y, f'{window[y, x]:.0f}', ha='center', va='center', color=color, fontsize=11)

axes[0].add_patch(plt.Rectangle((center_x - 1.5, center_y - 1.5), 3, 3, edgecolor='cyan', facecolor='none', linewidth=2))

axes[1].text(0.5, -0.18, f'Max value = {window_max:.0f}', transform=axes[1].transAxes, ha='center', fontsize=10)
axes[2].add_patch(plt.Rectangle((center_x - 0.5, center_y - 0.5), 1, 1, edgecolor='cyan', facecolor='none', linewidth=2))

plt.tight_layout()

Now lets look at the output of the maximum filter on our peak image. This will use the `find_local_maxima()` function defined. Go read that now and then execute the next code cell.

In [ ]:
# Detect candidate peaks at pixel resolution.
peak_threshold = log_filtered.mean() + 2.8 * log_filtered.std()
max_filtered = ndimage.maximum_filter(log_filtered, size=9, mode='reflect')
peaks = find_local_maxima(log_filtered, min_distance=4, threshold=peak_threshold)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(log_filtered, cmap='magma')
axes[0].set_title('LoG-filtered image')
axes[0].axis('off')

axes[1].imshow(max_filtered, cmap='magma')
axes[1].set_title('Maximum-filtered image')
axes[1].axis('off')

axes[2].imshow(demo_image)
axes[2].scatter(peaks[:, 1], peaks[:, 0], s=90, facecolors='none', edgecolors='C3', linewidths=2)
axes[2].set_title('Pixel-resolution peaks')
axes[2].axis('off')

plt.tight_layout()
print('Detected peaks (row, col):')
print(peaks)

## Subpixel Localization by Centroid

The centroid method computes an intensity-weighted average of nearby pixel coordinates:

$$
x_c = \frac{\sum x I(x, y)}{\sum I(x, y)}, \qquad y_c = \frac{\sum y I(x, y)}{\sum I(x, y)}.
$$

It is simple and fast, and often gives a noticeable improvement over using the single brightest pixel.

In [ ]:
peak_y, peak_x = peaks[0]
centroid_y, centroid_x = centroid_localization(demo_image, peak_y, peak_x, radius=5)

radius = 5
y0 = max(0, peak_y - radius)
y1 = min(demo_image.shape[0], peak_y + radius + 1)
x0 = max(0, peak_x - radius)
x1 = min(demo_image.shape[1], peak_x + radius + 1)
patch = demo_image[y0:y1, x0:x1]

plt.figure(figsize=(4.5, 4.5))
plt.imshow(patch)
plt.scatter(peak_x - x0, peak_y - y0, marker='x', s=140, color='C3', label='Peak pixel')
plt.scatter(centroid_x - x0, centroid_y - y0, marker='+', s=160, color='C0', label='Centroid')
plt.legend(loc='upper right')
plt.title('Centroid localization on one particle')
plt.axis('off')
plt.tight_layout()

print(f'Peak pixel: ({peak_y}, {peak_x})')
print(f'Centroid  : ({centroid_y:.2f}, {centroid_x:.2f})')

### Exercise 1

Change the peak used in the code cell above. Can you see from the image why the centroid is placed where it is?

## Subpixel Localization by Gaussian Fitting

A more model-based approach is to fit the particle image to a 2D Gaussian. Recall, the PSF Airy Disc is very similar to a Gaussian function in shape near the center. This is usually more accurate than the centroid when the signal-to-noise ratio is good, because the fit uses the whole local intensity profile rather than only a weighted average.

In [ ]:
fit_result = fit_gaussian_2d(demo_image, int(peak_y), int(peak_x), radius=5)
fit_y = fit_result['y']
fit_x = fit_result['x']

patch_y0 = fit_result['yy'][0, 0]
patch_x0 = fit_result['xx'][0, 0]

plt.figure(figsize=(4.8, 4.8))
plt.imshow(fit_result['patch'])
plt.scatter(peak_x - patch_x0, peak_y - patch_y0, marker='x', s=140, color='C3', label='Peak pixel')
plt.scatter(centroid_x - patch_x0, centroid_y - patch_y0, marker='+', s=150, color='C0', label='Centroid')
plt.scatter(fit_x - patch_x0, fit_y - patch_y0, marker='o', s=90, facecolors='none', edgecolors='C2', linewidths=2, label='Gaussian fit')
plt.title('Localization estimates')
plt.axis('off')
plt.legend(loc='upper right')
plt.tight_layout()

print(f'Peak pixel: ({peak_y}, {peak_x})')
print(f'Centroid  : ({centroid_y:.2f}, {centroid_x:.2f})')
print(f'Gaussian fit center: ({fit_y:.2f}, {fit_x:.2f})')
print(f'Estimated sigma   : {fit_result["sigma"]:.2f} pixels')

## Tracking Over Time

Once we have positions in each frame, the next step is to match detections between successive frames. This is a data-association problem: which particle in frame `t+1` is the continuation of each particle in frame `t`?

Let's load the simulated_particle_movie.tif and find all the particles in each frame.

In [ ]:
# Load a movie stack and localize particles in each time frame.
from tifffile import imread

movie_frames = imread(media_dir / 'simulated_particle_movie.tif')
localized_positions = []

for t in range(movie_frames.shape[0]):
    frame = movie_frames[t]
    response = -ndimage.gaussian_laplace(frame, sigma=1.5)
    threshold = response.mean() + 2.8 * response.std()
    peaks_t = find_local_maxima(response, min_distance=4, threshold=threshold)

    refined_t = []
    for y, x in peaks_t:
        refined_position = centroid_localization(frame, int(y), int(x), radius=4)
        refined_t.append(refined_position)
    refined_t = np.array(refined_t)

    localized_positions.append(refined_t)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))
for ax, frame_idx in zip(axes, [0, movie_frames.shape[0] - 1]): #loop over the first and last frame
    ax.imshow(movie_frames[frame_idx])
    pts = localized_positions[frame_idx]
    ax.scatter(pts[:, 1], pts[:, 0], s=90, facecolors='none', edgecolors='C3', linewidths=2)
    ax.set_title(f'Frame {frame_idx}')
    ax.axis('off')
plt.tight_layout()

print('Movie stack shape (time, y, x):', movie_frames.shape)
print('Localized particles per frame:', [len(p) for p in localized_positions])

## Nearest-Neighbor Matching

The simplest linker is greedy nearest-neighbor matching: for each particle in frame `t`, find the closest particle in frame `t+1`.

This works well when motion is small and particles are sparse.

In [ ]:
# Match two successive frames with a simple nearest-neighbor search.
points_t = localized_positions[0]
points_t1 = localized_positions[1]

matches = []
for i in range(len(points_t)):
    best_j = -1
    best_distance = np.inf

    for j in range(len(points_t1)):
        dy = points_t[i, 0] - points_t1[j, 0]
        dx = points_t[i, 1] - points_t1[j, 1]
        distance = np.sqrt(dx**2 + dy**2)

        if distance < best_distance:
            best_distance = distance
            best_j = j

    matches.append((i, best_j, best_distance))

plt.figure(figsize=(5, 5))
plt.imshow(movie_frames[0])
plt.scatter(points_t[:, 1], points_t[:, 0], color='C0', label='Frame 0')
plt.scatter(points_t1[:, 1], points_t1[:, 0], color='C1', label='Frame 1')

for i, j, distance in matches:
    plt.plot([points_t[i, 1], points_t1[j, 1]], [points_t[i, 0], points_t1[j, 0]], color='white', linewidth=1.5)

plt.legend(loc='lower left')
plt.title('Nearest-neighbor matches between frames 0 and 1')
plt.axis('off')
plt.tight_layout()

for i, j, distance in matches:
    print(f'Particle {i} -> {j}, distance = {distance:.2f}')

## Global Assignment with the Hungarian Algorithm

Nearest-neighbor matching can fail when particles get close together. A better strategy is to build a full *cost matrix* of pairwise distances and find the one-to-one assignment that minimizes the **total** cost (distance).

In Python, `scipy.optimize.linear_sum_assignment` solves this optimal assignment problem.

In [ ]:
# Solve the same frame-to-frame association problem globally.
cost_matrix = compute_cost_matrix(points_t, points_t1)
row_ind, col_ind = linear_sum_assignment(cost_matrix)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
im = axes[0].imshow(cost_matrix, cmap='Blues')
axes[0].set_title('Distance matrix')
axes[0].set_xlabel('Particles in frame 1')
axes[0].set_ylabel('Particles in frame 0')
for i in range(cost_matrix.shape[0]):
    for j in range(cost_matrix.shape[1]):
        axes[0].text(j, i, f'{cost_matrix[i, j]:.1f}', ha='center', va='center', color='black')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].imshow(movie_frames[0])
axes[1].scatter(points_t[:, 1], points_t[:, 0], color='C0', label='Frame 0')
axes[1].scatter(points_t1[:, 1], points_t1[:, 0], color='C1', label='Frame 1')
for i, j in zip(row_ind, col_ind):
    axes[1].plot([points_t[i, 1], points_t1[j, 1]], [points_t[i, 0], points_t1[j, 0]], color='white', linewidth=1.5)
axes[1].set_title('Hungarian assignments')
axes[1].axis('off')
axes[1].legend(loc='lower left')
plt.tight_layout()

print('Assignments (frame 0 index -> frame 1 index):')
for i, j in zip(row_ind, col_ind):
    print(f'{int(i)} -> {int(j)}')

Now, let's track the whole movie.

In [ ]:
# Extend Hungarian matching across the whole movie to build simple tracks.
tracks = {}
for track_id in range(len(localized_positions[0])):
    tracks[track_id] = [localized_positions[0][track_id]]

prev_points = localized_positions[0]
prev_ids = np.arange(len(prev_points))
next_track_id = len(prev_points)

for frame_idx in range(1, len(localized_positions)):
    current_points = localized_positions[frame_idx]
    cost = compute_cost_matrix(prev_points, current_points)
    row_ind, col_ind = linear_sum_assignment(cost)

    new_ids = np.full(len(current_points), -1, dtype=int)
    for r, c in zip(row_ind, col_ind):
        track_id = prev_ids[r]
        tracks[track_id].append(current_points[c])
        new_ids[c] = track_id

    for c in range(len(current_points)):
        if new_ids[c] == -1:
            tracks[next_track_id] = [current_points[c]]
            new_ids[c] = next_track_id
            next_track_id += 1

    prev_points = current_points
    prev_ids = new_ids

plt.figure(figsize=(5.5, 5.5))
plt.imshow(movie_frames[0])
for track_id, pts in tracks.items():
    pts = np.array(pts)
    plt.plot(pts[:, 1], pts[:, 0], '-o', linewidth=1.5, markersize=4, label=f'Track {track_id}')
plt.title(f'Tracked trajectories over {movie_frames.shape[0]} frames')
plt.axis('off')
plt.tight_layout()

## Beyond Simple Frame-to-Frame Linking

Pushing further...

- **Temporal models** such as Kalman filters can help when particles cross, disappear briefly, or move in a predictable way.
- **TrackPy** is a full particle-tracking toolkit that packages many of these ideas into a practical workflow.
- If the signal is not naturally sparse and point-like, **optical flow** or **particle image velocimetry (PIV)** may be a better representation of motion than explicit particle linking. *I think you will do this in precept.*